# TP final · Opción B · EDA y Limpieza de datos

## Análisis exploratorio de datos

Problema: predecir `Music effects` (`Improve`, `No effect`, `Worsen`) a partir de los hábitos de escucha reportados en la encuesta.

Este notebook es el análisis exploratorio. Carga la encuesta con Spark y la describe, sin imputar, sin descartar filas y sin entrenar. El objetivo es decidir después qué arreglo merece la tabla. Para eso mira:

- Cuántas filas y columnas hay, y qué significa cada columna
- Qué valores toma cada columna, de qué tipo es, cuántos hay de cada uno y cuántos faltan
- Cómo se distribuyen las columnas categóricas y las numéricas.
- En `Age`, `Hours per day` y `BPM`, el mínimo, el máximo y si hay vacíos o números que no se pueden leer.
- Qué filas de `BPM` quedan fuera de `40–250`. Ese rango no es una regla de la encuesta: sirve para ver tempos imposibles, como `0` o `999999999`.

Con esa salida se elige el arreglo. Los modelos y qué puntajes de salud entran como variables quedan para el notebook de entrenamiento.

**Fuente**: [Music & Mental Health Survey Results](https://www.kaggle.com/datasets/catherinerasgaitis/mxmh-survey-results), Catherine Rasgaitis, licencia CC0. El CSV está en `data/mxmh_survey_results.csv`.


## Widgets

- **Archivo CSV:** Ruta a los datos de la encuesta. Ya viene con la ruta del CSV de este repo.

In [ ]:
def ruta_csv_default():
    notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    carpeta = "/Workspace" + notebook.rsplit("/", 1)[0]
    return "file:" + carpeta + "/data/mxmh_survey_results.csv"

for nombre in ("csv_path", "puntajes_salud", "puntajes_algunas"):
    try:
        dbutils.widgets.remove(nombre)
    except Exception:
        pass

dbutils.widgets.text("csv_path", ruta_csv_default(), "Archivo CSV")

In [ ]:
from pyspark.sql import functions as F

def resolver_csv():
    pedido = dbutils.widgets.get("csv_path").strip()
    return pedido or ruta_csv_default()

csv_path = resolver_csv()
print("csv:", csv_path)

crudo = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)
print(f"filas: {crudo.count():,}  columnas: {len(crudo.columns)}")

## 1. Qué hay en la tabla

Conteos, tipo y nulos. Una celda vacía cuenta como faltante, igual que un nulo. 

En cada columna se listan hasta 8 valores, ordenados por frecuencia. 

Después el análisis sigue por columnas categóricas y por columnas numéricas.


In [ ]:
print("Primeras filas:")

display(crudo.limit(5))

TOPE = 8
filas = []
for nombre in crudo.columns:
    texto = F.trim(F.col(nombre).cast("string"))
    vacio = F.col(nombre).isNull() | (texto == "")
    presentes = crudo.where(~vacio)
    conteos = (
        presentes.groupBy(texto.alias("valor"))
        .count()
        .orderBy(F.desc("count"), "valor")
        .collect()
    )
    mostrados = conteos[:TOPE]
    partes = [str(fila.valor) for fila in mostrados]
    resto = len(conteos) - len(mostrados)
    if resto > 0:
        partes.append(f"y {resto} más")
    tipo = dict(crudo.dtypes)[nombre]
    filas.append((nombre, tipo, int(crudo.where(vacio).count()), len(conteos), ", ".join(partes)))

print("Resumen de columnas:")

display(spark.createDataFrame(filas, ["columna", "tipo", "nulos", "distintos", "ejemplo_valores"]))

`Timestamp` es la fecha y hora en que la persona envió el formulario. Aparece en la tabla de arriba y no tiene gráfico: casi no se repite.


In [ ]:
NUMERICOS = ("byte", "short", "int", "bigint", "long", "float", "double", "decimal")
numericas = [nombre for nombre, tipo in crudo.dtypes if tipo.startswith(NUMERICOS)]
categoricas = [nombre for nombre in crudo.columns if nombre not in numericas]
print("Numéricas:", ", ".join(numericas))
print("Categóricas:", ", ".join(categoricas))

## 2. Categóricas

Barras para las columnas de texto con pocos valores. `Timestamp` no entra en estos gráficos: casi cada fila tiene un valor distinto.

`Music effects` es la variable objetivo: si la persona dice si la música le mejora (`Improve`), no le hace nada (`No effect`) o le empeora (`Worsen`) su salud mental. El `(vacío)` son filas sin respuesta.

`Primary streaming service` es el servicio de música que más usa.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def conteo(columna):
    texto = F.trim(F.col(columna).cast("string"))
    vacio = F.col(columna).isNull() | (texto == "")
    return (
        crudo.select(F.when(vacio, "(vacío)").otherwise(texto).alias("valor"))
        .groupBy("valor")
        .count()
        .orderBy(F.desc("count"), "valor")
        .toPandas()
    )

def barras(tabla, titulo, ax, horizontal=False):
    if horizontal:
        orden = tabla.iloc[::-1]
        ax.barh(orden["valor"], orden["count"], color="#2563eb")
        for valor, cantidad in zip(orden["valor"], orden["count"]):
            ax.text(cantidad, valor, f" {int(cantidad)}", va="center", fontsize=8)
        ax.set_xlabel("filas")
    else:
        ax.bar(tabla["valor"], tabla["count"], color="#2563eb")
        ax.tick_params(axis="x", rotation=30)
        for i, cantidad in enumerate(tabla["count"]):
            ax.text(i, cantidad, str(int(cantidad)), ha="center", va="bottom", fontsize=8)
        ax.set_ylabel("filas")
    ax.set_title(titulo, fontsize=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
barras(conteo("Music effects"), "Music effects", axes[0])
barras(conteo("Primary streaming service"), "Primary streaming service", axes[1], horizontal=True)
fig.tight_layout()
plt.show()

## Variable objetivo

La tabla parte el mismo conteo de `Music effects`: filas y porcentaje sobre el total. `(vacío)` no es una clase. Son respuestas sin etiqueta.


In [ ]:
efectos = conteo("Music effects")
total = int(efectos["count"].sum())
efectos["filas"] = efectos["count"].astype(int)
efectos["porcentaje"] = (efectos["filas"] / total * 100).round(1)
orden = {"Improve": 0, "No effect": 1, "Worsen": 2, "(vacío)": 3}
efectos = efectos.sort_values("valor", key=lambda s: s.map(lambda v: orden.get(v, 9)))
display(efectos[["valor", "filas", "porcentaje"]].reset_index(drop=True))


`Improve` es la mayoría. `No effect` es la segunda clase. `Worsen` es un grupo chico. Un modelo que siempre contesta `Improve` acierta la mayor parte de las filas, y por eso la comparación usa el F1 macro y no la accuracy. Las filas vacías no se imputan: la clase no se inventa. El descarte queda para `01_experimentos.ipynb`.


`Fav genre` es el género favorito de la persona.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
barras(conteo("Fav genre"), "Fav genre", ax, horizontal=True)
fig.tight_layout()
plt.show()

Esas seis columnas son binarias (*sí*/*no*), salvo `Permissions`:

- `While working`: si escucha música mientras trabaja.
- `Instrumentalist`: si toca un instrumento.
- `Composer`: si compone música.
- `Exploratory`: si busca artistas o géneros nuevos.
- `Foreign languages`: si escucha música en otros idiomas.
- `Permissions`: si aceptó que se publique su respuesta. No describe un hábito de escucha.


In [ ]:
binarias = [
    "While working", "Instrumentalist", "Composer",
    "Exploratory", "Foreign languages", "Permissions",
]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, columna in zip(axes.ravel(), binarias):
    barras(conteo(columna), columna, ax)
fig.tight_layout()
plt.show()

### Mapa de calor - frecuencia de escucha por género musical

Cada fila es una columna `Frequency [...]`: con qué frecuencia se escucha ese género. Los valores son `Never`, `Rarely`, `Sometimes` y `Very frequently`.

Las columnas son `Classical`, `Country`, `EDM`, `Folk`, `Gospel`, `Hip hop`, `Jazz`, `K pop`, `Latin`, `Lofi`, `Metal`, `Pop`, `R&B`, `Rap`, `Rock` y `Video game music`.


In [ ]:
frecuencias = [nombre for nombre in categoricas if nombre.startswith("Frequency [")]
niveles = ["Never", "Rarely", "Sometimes", "Very frequently", "(vacío)"]
matriz = []
for columna in frecuencias:
    tabla = conteo(columna).set_index("valor")["count"]
    matriz.append([int(tabla.get(nivel, 0)) for nivel in niveles])
fig, ax = plt.subplots(figsize=(8, 6))
imagen = ax.imshow(np.array(matriz), cmap="Blues", aspect="auto")
ax.set_xticks(range(len(niveles)), niveles, rotation=30, ha="right")
ax.set_yticks(range(len(frecuencias)), [nombre.removeprefix("Frequency ").strip("[]") for nombre in frecuencias])
ax.set_title("Frecuencia por género musical", fontsize=10)
for i in range(len(frecuencias)):
    for j in range(len(niveles)):
        ax.text(j, i, matriz[i][j], ha="center", va="center", fontsize=7, color="black")
fig.colorbar(imagen, ax=ax, label="filas")
fig.tight_layout()
plt.show()

### Mapa de calor - hábito por género y Music effects

El mapa de arriba no mira la etiqueta. Este cruza cada género con `Music effects`. El número es el hábito medio de esa clase: `Never` vale 0, `Rarely` 1, `Sometimes` 2 y `Very frequently` 3. No entran filas con el género vacío ni con `Music effects` vacío.


In [ ]:
puntos = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Very frequently": 3}
clases = ["Improve", "No effect", "Worsen"]
columnas = ["Music effects", *frecuencias]
pdf = crudo.select(*[F.trim(F.col(c).cast("string")).alias(c) for c in columnas]).toPandas()
pdf = pdf[pdf["Music effects"].isin(clases)]
generos = [nombre.removeprefix("Frequency ").strip("[]") for nombre in frecuencias]
matriz = []
for columna in frecuencias:
    fila = []
    for clase in clases:
        habito = pdf.loc[pdf["Music effects"] == clase, columna].map(puntos)
        fila.append(float(habito.mean()) if habito.notna().any() else float("nan"))
    matriz.append(fila)
fig, ax = plt.subplots(figsize=(6, 6))
imagen = ax.imshow(np.array(matriz), cmap="Blues", aspect="auto", vmin=0, vmax=3)
ax.set_xticks(range(len(clases)), clases, rotation=30, ha="right")
ax.set_yticks(range(len(generos)), generos)
ax.set_title("Hábito medio por género y por Music effects", fontsize=10)
for i in range(len(generos)):
    for j in range(len(clases)):
        ax.text(j, i, f"{matriz[i][j]:.2f}", ha="center", va="center", fontsize=8, color="black")
fig.colorbar(imagen, ax=ax, label="0 Never · 3 Very frequently")
fig.tight_layout()
plt.show()


Un número más alto quiere decir que esa clase escucha ese género más seguido. `Worsen` tiene 17 filas, así que su columna se mueve con pocas personas. El mapa no descarta ni imputa nada.


## 3. Numéricas


- `Age`: edad.
- `Hours per day`: horas por día que escucha música.
- `BPM`: (*Beats Per Minute*) pulsos por minuto que sintió.
- `Anxiety`, `Depression`, `Insomnia` y `OCD`: Puntaje de salud mental autopercibidos de `0` a `10`.
  - `0` es no experimentarlo 
  - `10` es experimentarlo de forma constante o extrema.

In [ ]:
def resumen_numerico(nombre):
    texto = F.trim(F.col(nombre).cast("string"))
    numero = F.col(nombre).cast("double")
    vacio = F.col(nombre).isNull() | (texto == "")
    no_numerico = ~vacio & numero.isNull()
    print(nombre)
    display(crudo.select(
        F.count("*").alias("filas"),
        F.count(F.when(vacio, 1)).alias("vacios"),
        F.count(F.when(no_numerico, 1)).alias("no_numerico"),
        F.min(numero).alias("minimo"),
        F.avg(numero).alias("promedio"),
        F.max(numero).alias("maximo"),
    ))

for nombre in numericas:
    resumen_numerico(nombre)

bpm = F.col("BPM").cast("double")

In [ ]:
def histograma(nombre, ax, techo=None):
    numero = F.col(nombre).cast("double")
    serie = crudo.select(numero.alias("valor")).where(F.col("valor").isNotNull())
    if techo is not None:
        serie = serie.where(F.col("valor") <= techo)
    valores = [fila.valor for fila in serie.collect()]
    ax.hist(valores, bins=20, color="#2563eb")
    ax.set_title(nombre, fontsize=10)
    ax.set_ylabel("filas")

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
nombres = [n for n in numericas if n != "BPM"]
for ax, nombre in zip(axes.ravel(), nombres):
    histograma(nombre, ax)
for ax in axes.ravel()[len(nombres):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
histograma("BPM", ax, techo=300)
ax.set_title("BPM hasta 300 (el máximo real queda afuera del eje)")
fig.tight_layout()
plt.show()

In [ ]:
print("Filas cuyo BPM está vacío, no es un número, o cae fuera de 40-250. No modifica la tabla.")
raros = crudo.where(
    F.col("BPM").isNull()
    | (F.trim(F.col("BPM").cast("string")) == "")
    | bpm.isNull()
    | (bpm < 40)
    | (bpm > 250)
)
print(f"filas con BPM vacio, no numerico o fuera de 40-250: {raros.count()}")
display(raros.select("Age", "Fav genre", "BPM", "Music effects").orderBy(bpm.asc_nulls_first()))

El rango 40–250 no viene de la encuesta. Un tema suele ir de 60 a 180, y 40–250 es un margen ancho para ver tempos que no son de una canción, como 0 o 999999999. La lista de arriba muestra esas filas. Todavía no es una regla de limpieza: el arreglo se decide después de mirarlas.


## Conclusión

La encuesta tiene 736 filas y 33 columnas. La variable objetivo `Music effects` está desbalanceada: la mayoría dice `Improve`, `Worsen` es un grupo chico y 8 filas no tienen respuesta. Esas 8 no se pueden imputar, porque la clase no se inventa.

En el resto, los faltantes son pocos salvo en `BPM`: 107 vacíos, un máximo de 999999999, 6 valores por debajo de 40 y 2 por encima de 250. `Age` tiene un vacío. `Primary streaming service`, `While working`, `Instrumentalist`, `Composer` y `Foreign languages` tienen entre 1 y 4 vacíos.

El arreglo de filas queda para la próxima decisión: qué hacer con las filas sin etiqueta, con los `BPM` vacíos o imposibles, y con los pocos vacíos de las demás columnas. Esta exploración no imputa ni descarta filas. La limpieza de abajo solo ajusta tipos y saca `Permissions`.


## Limpieza

Se arma `survey_cleaned` a partir de `crudo` y se guarda en `bigdata_emanuel_rb.survey_cleaned`. `crudo` queda como se leyó el CSV.

- `Timestamp` pasa a timestamp, con hora.
- `Age` y los puntajes de salud pasan a entero.
- `Hours per day` y `BPM` pasan a double.
- Las columnas sí/no quedan como string (`Yes`, `No` o vacío).
- El resto queda como string.
- Antes de guardar, los nombres pasan a minúsculas y guion bajo. Delta no acepta espacios ni corchetes. `Music effects` queda `music_effects` y `Frequency [Rock]` queda `frequency_rock`.
- `Permissions` se elimina: solo dice si la persona aceptó publicar la respuesta. No describe un hábito ni la etiqueta.
- En `Primary streaming service`, `I do not use a streaming service` pasa a `No streaming` y `Other streaming service` pasa a `Other`.

No se imputa y no se descarta ninguna fila.


In [ ]:
enteros = ["Age", "Anxiety", "Depression", "Insomnia", "OCD"]
decimales = ["Hours per day", "BPM"]

survey_cleaned = crudo.drop("Permissions")
survey_cleaned = survey_cleaned.withColumn("Timestamp", F.to_timestamp("Timestamp", "M/d/yyyy H:mm:ss"))
for nombre in enteros:
    survey_cleaned = survey_cleaned.withColumn(nombre, F.col(nombre).cast("int"))
for nombre in decimales:
    survey_cleaned = survey_cleaned.withColumn(nombre, F.col(nombre).cast("double"))
survey_cleaned = survey_cleaned.withColumn(
    "Primary streaming service",
    F.when(F.col("Primary streaming service") == "I do not use a streaming service", "No streaming")
     .when(F.col("Primary streaming service") == "Other streaming service", "Other")
     .otherwise(F.col("Primary streaming service")),
)

def nombre_delta(nombre):
    texto = nombre.lower().replace("&", "").replace("[", "").replace("]", "")
    return "_".join(texto.split())

survey_cleaned = survey_cleaned.toDF(*[nombre_delta(nombre) for nombre in survey_cleaned.columns])

catalogo = spark.sql("SELECT current_catalog()").first()[0]
tabla = f"{catalogo}.bigdata_emanuel_rb.survey_cleaned"
survey_cleaned.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
print(f"filas: {survey_cleaned.count():,}  columnas: {len(survey_cleaned.columns)}")
print("tabla:", tabla)
survey_cleaned.printSchema()
